# English Lecture Audio to STT & Summarization Pipeline
This notebook runs on Google Colab (with GPU) using OpenAI's Whisper model to transcribe English lecture audio files dynamically, followed by English and Korean summarization using Hugging Face transformers.

In [ ]:
# 1. Install necessary libraries for audio transcription and summarization
!pip install -q openai-whisper ffmpeg-python transformers torch
import whisper
import os
import torch
from google.colab import files
from transformers import pipeline

print('Libraries loaded successfully!')

In [ ]:
# 2. Dynamic Audio File Upload (Runs interactively in Google Colab)
print('Please upload your lecture audio file (e.g., .m4a, .mp3, .wav):')
uploaded = files.upload()

# Automatically grab the uploaded filename and set dynamic output path
input_audio_path = list(uploaded.keys())[0]
base_name = os.path.splitext(input_audio_path)[0]
output_txt_path = f"{base_name}-stt.txt"
output_summary_path = f"{base_name}-summary.txt"

print(f"\nTarget Input Audio: {input_audio_path}")
print(f"Target Output Text: {output_txt_path}")

In [ ]:
# 3. Load Whisper Model and Transcribe (Using GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Model sizes: 'tiny', 'base', 'small', 'medium', 'large'
model_size = "base" 
print(f"Loading Whisper model ({model_size})...")
model = whisper.load_model(model_size, device=device)

print("Transcribing audio... (this will be fast with GPU)")
result = model.transcribe(input_audio_path, language="en")

transcript_text = result["text"]
print("Transcription completed!")

In [ ]:
# 4. Generate English and Korean Summaries using Transformers
print("Generating English Summary...")
# Using BART-large-cnn for abstractive summarization
summarizer_en = pipeline("summarization", model="facebook/bart-large-cnn", device=0 if torch.cuda.is_available() else -1)

# Handle long text by chunking if necessary (BART max token limit is typically 1024)
max_chunk_length = 1000
chunks = [transcript_text[i:i+max_chunk_length] for i in range(0, len(transcript_text), max_chunk_length)]

english_summaries = []
for chunk in chunks:
    if len(chunk.strip()) > 50:
        summary = summarizer_en(chunk, max_length=150, min_length=30, do_sample=False)
        english_summaries.append(summary[0]['summary_text'])

english_summary_text = "\n".join(english_summaries)

# Generate Korean Summary using a multilingual translation/summarization model or pipeline
print("Generating Korean Summary...")
try:
    # Using a translation model to translate English summary to Korean
    translator = pipeline("translation_en_to_ko", model="Helsinki-NLP/opus-mt-en-ko", device=0 if torch.cuda.is_available() else -1)
    korean_summary_chunks = []
    for chunk_text in english_summaries:
        translated = translator(chunk_text, max_length=200)
        korean_summary_chunks.append(translated[0]['translation_text'])
    korean_summary_text = "\n".join(korean_summary_chunks)
except Exception as e:
    korean_summary_text = f"Translation fallback error: {str(e)}\nEnglish Summary:\n{english_summary_text}"

print("Summarization completed!")

In [ ]:
# 5. Save and Download Results (STT + Summaries)
with open(output_txt_path, "w", encoding="utf-8") as f:
    f.write(transcript_text)

with open(output_summary_path, "w", encoding="utf-8") as f:
    f.write("=== ENGLISH SUMMARY ===\n")
    f.write(english_summary_text)
    f.write("\n\n=== KOREAN SUMMARY ===\n")
    f.write(korean_summary_text)

print(f"Successfully saved STT to: {output_txt_path}")
print(f"Successfully saved Summary to: {output_summary_path}")

# Trigger browser downloads
files.download(output_txt_path)
files.download(output_summary_path)

# Display previews
print("\n--- English Summary Preview ---")
print(english_summary_text[:500])
print("\n--- Korean Summary Preview ---")
print(korean_summary_text[:500])